# beat_this a ONNX

Convierte el detector de pulso beat_this (CPJKU, licencia MIT) a ONNX, para correrlo en la Mac con onnxruntime, que ya usa basic-pitch, sin instalar PyTorch.

Devuelve `beat-this-onnx.zip` con:
- `beat_this.onnx`: el modelo, que recibe trozos de 1500 cuadros de espectrograma (30 s).
- `fuentes/`: el código de preprocesado, troceo y posprocesado de beat_this, para copiarlo igual en la Mac.
- `referencia.npz`: un audio de prueba con su espectrograma, salidas del modelo y pulsos, para comprobar que la versión de la Mac da lo mismo.

No necesita GPU. Entorno de ejecución > Ejecutar todas; al final se descarga el zip solo.

In [ ]:
!pip -q install "git+https://github.com/CPJKU/beat_this.git" einops rotary-embedding-torch soxr onnx onnxruntime onnxscript

In [ ]:
import inspect, os, beat_this, beat_this.preprocessing, beat_this.inference, beat_this.model.postprocessor
# Se guardan las fuentes tal cual: los parámetros del espectrograma y del troceo tienen que
# copiarse exactos, y leerlos del código evita adivinarlos.
os.makedirs("fuentes", exist_ok=True)
for mod in (beat_this.preprocessing, beat_this.inference, beat_this.model.postprocessor):
    with open(f"fuentes/{mod.__name__}.py", "w") as f:
        f.write(inspect.getsource(mod))
print(os.listdir("fuentes"))

In [ ]:
import torch
from beat_this.inference import load_model

model = load_model("final0", device="cpu").eval()

class Logits(torch.nn.Module):
    """El modelo devuelve un diccionario; ONNX necesita salidas con nombre fijo."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, spect):
        out = self.model(spect)
        return out["beat"], out["downbeat"]

wrapped = Logits(model).eval()
dummy = torch.randn(1, 1500, 128)
torch.onnx.export(wrapped, (dummy,), "beat_this.onnx", input_names=["spect"],
                  output_names=["beat", "downbeat"], opset_version=17, dynamo=False)
print(round(os.path.getsize("beat_this.onnx") / 1e6, 1), "MB")

In [ ]:
import numpy as np, onnxruntime as ort, librosa
from beat_this.inference import File2Beats, Audio2Frames
from beat_this.preprocessing import load_audio

# Audio de prueba: un fragmento real de GuitarSet, para que el modelo encuentre pulsos.
!wget -q -O mic.zip "https://zenodo.org/api/records/3371780/files/audio_mono-mic.zip/content"
!unzip -q -o mic.zip "*00_BN1-129-Eb_comp_mic.wav" "*00_SS2-88-F_comp_mic.wav" -d ref && rm mic.zip
import glob
wavs = sorted(glob.glob("ref/**/*.wav", recursive=True))
print(wavs)

sess = ort.InferenceSession("beat_this.onnx")
ref = {}
f2b = File2Beats(checkpoint_path="final0", device="cpu", dbn=False)
for i, wav in enumerate(wavs):
    signal, sr = load_audio(wav)
    # Igual que Audio2Frames: mono y a 22050 Hz antes del espectrograma.
    if signal.ndim == 2:
        signal = signal.mean(1)
    if sr != 22050:
        import soxr
        signal, sr = soxr.resample(signal, sr, 22050), 22050
    from beat_this.preprocessing import LogMelSpect
    spect = getattr(f2b, "spect", None) or LogMelSpect()
    spect = spect(torch.tensor(signal, dtype=torch.float32))
    beats, downbeats = f2b(wav)
    ref[f"audio{i}"] = signal.astype(np.float32); ref[f"sr{i}"] = sr
    ref[f"beats{i}"] = np.asarray(beats); ref[f"downbeats{i}"] = np.asarray(downbeats)
    if True:
        spect = spect.numpy()
        ref[f"spect{i}"] = spect
        chunk = spect[None, :1500]
        with torch.no_grad():
            tb, td = wrapped(torch.tensor(chunk))
        ob, od = sess.run(None, {"spect": chunk.astype(np.float32)})
        print(os.path.basename(wav), "diferencia torch/onnx", float(np.abs(tb.numpy() - ob).max()), float(np.abs(td.numpy() - od).max()))
        ref[f"logits{i}"] = tb.numpy()
    print(os.path.basename(wav), len(beats), "pulsos")
np.savez_compressed("referencia.npz", **ref)

In [ ]:
!zip -q -r beat-this-onnx.zip beat_this.onnx fuentes referencia.npz
from google.colab import files
files.download("beat-this-onnx.zip")